In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

X, y = make_moons(n_samples=300, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

dt_unpruned = DecisionTreeClassifier(random_state=42)
dt_unpruned.fit(X_train, y_train)

dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_depth3.fit(X_train, y_train)

print("=== Unconstrained vs. Depth-Controlled Tree ===")
print(f"Unpruned Tree  -> Train Acc: {accuracy_score(y_train, dt_unpruned.predict(X_train)):.4f}, "
      f"Test Acc: {accuracy_score(y_test, dt_unpruned.predict(X_test)):.4f}, Depth: {dt_unpruned.get_depth()}")
print(f"Max Depth = 3  -> Train Acc: {accuracy_score(y_train, dt_depth3.predict(X_train)):.4f}, "
      f"Test Acc: {accuracy_score(y_test, dt_depth3.predict(X_test)):.4f}, Depth: {dt_depth3.get_depth()}")

# 3. Cost-Complexity Pruning (ccp_alpha) analysis
path = dt_unpruned.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]  # Exclude trivial single-node root tree

train_scores, test_scores = [], []
clfs = []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    clfs.append(clf)
    train_scores.append(accuracy_score(y_train, clf.predict(X_train)))
    test_scores.append(accuracy_score(y_test, clf.predict(X_test)))

best_idx = int(np.argmax(test_scores))
best_alpha = ccp_alphas[best_idx]
dt_pruned = clfs[best_idx]

print(f"\n=== Cost-Complexity Pruning ===")
print(f"Best ccp_alpha: {best_alpha:.5f} -> Train Acc: {train_scores[best_idx]:.4f}, "
      f"Test Acc: {test_scores[best_idx]:.4f}, Depth: {dt_pruned.get_depth()}")

# 4. Visualize the pruned decision tree
plt.figure(figsize=(14, 6))
plot_tree(
    dt_pruned,
    feature_names=["Feature 1", "Feature 2"],
    class_names=["Class 0", "Class 1"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title(f"Pruned Decision Tree (ccp_alpha = {best_alpha:.4f})")
plt.tight_layout()
plt.show()

# 5. Plot Overfitting across Max Depth & Pruning Alpha
depths = range(1, 16)
depth_train_acc = [
    accuracy_score(y_train, DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train).predict(X_train))
    for d in depths
]
depth_test_acc = [
    accuracy_score(y_test, DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train).predict(X_test))
    for d in depths
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(depths, depth_train_acc, marker="o", label="Train Accuracy")
axes[0].plot(depths, depth_test_acc, marker="s", label="Test Accuracy")
axes[0].set_title("Overfitting vs. Tree Max Depth")
axes[0].set_xlabel("max_depth")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.4)

axes[1].plot(ccp_alphas, train_scores, marker="o", drawstyle="steps-post", label="Train Accuracy")
axes[1].plot(ccp_alphas, test_scores, marker="s", drawstyle="steps-post", label="Test Accuracy")
axes[1].axvline(best_alpha, color="crimson", linestyle="--", label=f"Best α = {best_alpha:.4f}")
axes[1].set_title("Accuracy vs. Pruning Parameter (ccp_alpha)")
axes[1].set_xlabel("ccp_alpha")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

# 6. Compare Decision Boundaries: Unpruned vs. Pruned
def plot_decision_boundary(model, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", edgecolors="k", s=35)
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(dt_unpruned, axes[0], "Unpruned Tree (Overfitted Boundary)")
plot_decision_boundary(dt_pruned, axes[1], f"Pruned Tree (α = {best_alpha:.4f})")
plt.tight_layout()
plt.show()